# Explorador interactivo de curvas de convergencia

Elige **red, optimizador, fitness, % de reglas, muestreo y métrica**, y el notebook
dibuja solo esas curvas. Sin forward-fill: cada curva llega hasta su última
generación real.

1. Ejecuta la celda de carga.
2. Ejecuta la celda de controles y juega con los selectores.

> Requiere `ipywidgets` (`pip install ipywidgets`). Ejecuta con Jupyter Lab/Notebook.

In [ ]:
import glob, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as W
from IPython.display import display

# Carpetas con curvas: etiqueta legible -> ruta. Anade aqui mas si quieres.
SOURCES = {
    'bypass (grid_raw)':   'example/bypass2/grid_raw',
    'bypass (grid_full)':  'example/bypass2/grid_full',
    'nhlv1 (grid_full)':   'example/nhlv1/grid_full_nhlv1',
}

def _resolve(p):
    # Permite ejecutar el notebook desde la raiz del repo o desde memoria/.
    for base in ['', '..', '../..']:
        cand = os.path.join(base, p) if base else p
        if glob.glob(os.path.join(cand, 'grid_search_curves_*.csv')):
            return cand
    return p

def load_curves():
    rows = []
    for label, path in SOURCES.items():
        path = _resolve(path)
        for f in sorted(glob.glob(os.path.join(path, 'grid_search_curves_*.csv'))):
            opt = os.path.basename(f).replace('grid_search_curves_', '').replace('.csv', '').upper()
            d = pd.read_csv(f)
            if d.empty:
                continue
            d['source'] = label
            d['optimizer'] = opt
            rows.append(d)
    if not rows:
        raise FileNotFoundError('No se encontraron grid_search_curves_*.csv. Ejecuta desde la raiz del repo.')
    return pd.concat(rows, ignore_index=True)

DF = load_curves()
METRICS = [c for c in ['mean_accuracy','mean_fitness','mean_error_chance','mean_error_utility',
                       'mean_entropy_norm','mean_util_dev','mean_gen_cpu'] if c in DF.columns]
print('Filas:', len(DF))
print('Fuentes:', list(DF['source'].unique()))
print('Optimizadores:', sorted(DF['optimizer'].unique()))
print('Metricas:', METRICS)
DF.head(3)

In [ ]:
PALETTE = ['#4C72B0','#55A868','#C44E52','#8172B3','#CCB974','#64B5CD','#E377C2','#7F7F7F']

# --- Controles ---
w_src   = W.Dropdown(options=list(DF['source'].unique()), description='Red/Fuente')
w_metric= W.Dropdown(options=METRICS, value='mean_accuracy', description='Metrica')
w_opt   = W.SelectMultiple(description='Optimizador', rows=5)
w_fit   = W.SelectMultiple(description='Fitness', rows=5)
w_pct   = W.SelectMultiple(description='% reglas', rows=4)
w_samp  = W.SelectMultiple(description='Muestreo', rows=2)
w_color = W.Dropdown(options=['optimizer','fitness_type','n_decision_rules_pct','sampling_mode'],
                     value='optimizer', description='Color por')
w_mark  = W.Checkbox(value=True, description='Marcadores')
w_merge = W.Checkbox(value=False, description='Promediar muestreos')
w_zoom  = W.Checkbox(value=False, description='Y desde 0')
out = W.Output()

def _populate(*_):
    # Rellena las opciones segun la fuente elegida.
    sub = DF[DF['source'] == w_src.value]
    w_opt.options  = sorted(sub['optimizer'].unique())
    w_fit.options  = sorted(sub['fitness_type'].unique())
    w_pct.options  = sorted(int(x) for x in sub['n_decision_rules_pct'].unique())
    w_samp.options = sorted(sub['sampling_mode'].unique())
    w_opt.value  = tuple(w_opt.options)
    w_fit.value  = (w_fit.options[0],) if w_fit.options else ()
    w_pct.value  = tuple(w_pct.options)
    w_samp.value = ('non_symmetric',) if 'non_symmetric' in w_samp.options else tuple(w_samp.options[:1])

def _plot(*_):
    with out:
        out.clear_output(wait=True)
        d = DF[(DF['source'] == w_src.value)
               & (DF['optimizer'].isin(w_opt.value))
               & (DF['fitness_type'].isin(w_fit.value))
               & (DF['n_decision_rules_pct'].isin(w_pct.value))
               & (DF['sampling_mode'].isin(w_samp.value))].copy()
        if d.empty:
            print('Sin curvas para esta seleccion.'); return
        m = w_metric.value
        group_cols = ['optimizer','fitness_type','n_decision_rules_pct','sampling_mode']
        if w_merge.value:
            group_cols.remove('sampling_mode')  # promedia los muestreos seleccionados
        # Color por la dimension elegida
        cby = w_color.value if w_color.value in group_cols else group_cols[0]
        cvals = sorted(d[cby].unique(), key=lambda v: (str(type(v)), v))
        cmap = {v: PALETTE[i % len(PALETTE)] for i, v in enumerate(cvals)}
        fig, ax = plt.subplots(figsize=(11, 6))
        for key, g in d.groupby(group_cols):
            key = key if isinstance(key, tuple) else (key,)
            info = dict(zip(group_cols, key))
            curve = g.groupby('generation')[m].mean()  # media real (reps/muestreos), sin ffill
            label = ' | '.join(f'{c}={info[c]}' for c in group_cols)
            ax.plot(curve.index.values, curve.values, color=cmap[info[cby]], lw=2.0,
                    marker='o' if w_mark.value else None, ms=3.5, label=label)
        ax.set_xlabel('Generacion'); ax.set_ylabel(m)
        if m == 'mean_accuracy':
            ax.set_ylim(0 if w_zoom.value else 45, 103)
        elif w_zoom.value:
            ax.set_ylim(bottom=0)
        ax.grid(alpha=.3)
        ax.set_title(f'{m}  ·  {w_src.value}  (color por {cby})')
        if len(ax.lines) <= 14:
            ax.legend(fontsize=8, loc='best')
        else:
            ax.text(0.99, 0.02, f'{len(ax.lines)} curvas (leyenda oculta)',
                    transform=ax.transAxes, ha='right', fontsize=9, color='gray')
        plt.tight_layout(); plt.show()

for w in [w_src, w_metric, w_opt, w_fit, w_pct, w_samp, w_color, w_mark, w_merge, w_zoom]:
    w.observe(_plot, names='value')
w_src.observe(_populate, names='value')

_populate(); _plot()
controls = W.VBox([
    W.HBox([w_src, w_metric, w_color]),
    W.HBox([w_opt, w_fit, w_pct, w_samp]),
    W.HBox([w_mark, w_merge, w_zoom]),
])
display(controls, out)